In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import scipy.sparse

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Clean CSV not found at: {csv_path}")
    df = pd.read_csv(csv_path)
    expected_cols = {"text", "label", "split"}
    if not expected_cols.issubset(set(df.columns)):
        raise ValueError(f"{csv_path} must contain columns {expected_cols}, but has {set(df.columns)}")
    return df.reset_index(drop=True)


def load_graph_npz(base_dir: str, dataset_name: str, k: int) -> Tuple[np.ndarray, np.ndarray, int]:
    graph_path = os.path.join(base_dir, "graphs", dataset_name, f"knn_k{k}.npz")
    if not os.path.exists(graph_path):
        raise FileNotFoundError(f"Graph file not found at: {graph_path}")
    try:
        sparse_matrix = scipy.sparse.load_npz(graph_path)
        row, col = sparse_matrix.nonzero()
        num_nodes = sparse_matrix.shape[0]
        return row, col, num_nodes
    except Exception as e:
        data = np.load(graph_path)
        if "row" in data and "col" in data:
            row, col = data["row"], data["col"]
        elif "sources" in data and "targets" in data:
            row, col = data["sources"], data["targets"]
        elif "indices" in data and "indptr" in data and "shape" in data:
            sm = scipy.sparse.csr_matrix(
                (np.ones(len(data["indices"])), data["indices"], data["indptr"]), shape=data["shape"]
            )
            row, col = sm.nonzero()
        else:
            raise ValueError(
                f"Graph file {graph_path} does not contain recognizable sparse-graph keys. "
                f"Keys found: {list(data.keys())}. Original error: {e}"
            )
        num_nodes = int(data["num_nodes"]) if "num_nodes" in data else int(max(row.max(), col.max()) + 1)
        return row, col, num_nodes


def encode_texts_for_dataset(df, tokenizer, transformer, device, max_length=128, batch_size=32) -> torch.Tensor:
    texts = df["text"].astype(str).tolist()
    all_embeddings = []
    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size:(i + 1) * batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)


def prepare_all_datasets(base_dir, model_name, max_length, batch_size, k, device, dataset_names):
    print(f"Using device: {device}")
    print(f"Transformer: {model_name}, k={k}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)

    all_data = {}
    for name in dataset_names:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")
        embeddings = encode_texts_for_dataset(df, tokenizer, transformer, device, max_length, batch_size)
        labels = torch.tensor(df["label"].values, dtype=torch.long)
        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)

        print("  Loading graph...")
        row, col, num_nodes = load_graph_npz(base_dir, name, k)
        if num_nodes != len(df):
            print(f"  [Warning] num_nodes ({num_nodes}) != len(df) ({len(df)}); clipping indices.")
            num_nodes = len(df)
            row = np.clip(row, 0, num_nodes - 1)
            col = np.clip(col, 0, num_nodes - 1)
        edge_index = torch.tensor(np.vstack([row, col]), dtype=torch.long)

        real_frac = (df.loc[df["split"] == "train", "label"] == 1).mean()
        print(f"  Train real fraction: {real_frac:.4f}")

        all_data[name] = {
            "df": df, "emb": embeddings, "labels": labels,
            "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
            "edge_index": edge_index, "num_nodes": num_nodes,
        }
    return all_data


# ================================================================
# Models: T-Base (no graph) and Hybrid (GCN), both with the fixes
# established in the sweep work: class weighting, grad clipping,
# LR warmup+decay.
# ================================================================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.0):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        row, col = edge_index
        device = x.device
        self_loop = torch.arange(num_nodes, device=device)
        row = torch.cat([row, self_loop], dim=0)
        col = torch.cat([col, self_loop], dim=0)
        deg = torch.zeros(num_nodes, device=device)
        deg.index_add_(0, row, torch.ones_like(row, dtype=deg.dtype, device=device))
        deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)
        x = self.linear(x)
        x = self.dropout(x)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        x_col = x[col] * norm.unsqueeze(-1)
        out = torch.zeros_like(x)
        out.index_add_(0, row, x_col)
        return out


class HybridGCNTransformer(nn.Module):
    def __init__(self, emb_dim, gcn_hidden=256, fusion_hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.gcn1 = GCNLayer(emb_dim, gcn_hidden, dropout=dropout)
        self.gcn2 = GCNLayer(gcn_hidden, gcn_hidden, dropout=dropout)
        self.fc_fuse = nn.Linear(emb_dim + gcn_hidden, fusion_hidden)
        self.fc_out = nn.Linear(fusion_hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        g = F.relu(self.gcn1(x, edge_index, num_nodes))
        g = F.relu(self.gcn2(g, edge_index, num_nodes))
        fused = torch.cat([x, g], dim=-1)
        fused = F.relu(self.fc_fuse(fused))
        fused = self.dropout(fused)
        return self.fc_out(fused)


class TextOnlyClassifier(nn.Module):
    def __init__(self, emb_dim, hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, *_ignored):  # accepts extra args so it can share a call signature with the hybrid model
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)


def compute_class_weights(y_src, mask, num_classes, device):
    labels_subset = y_src[mask]
    counts = torch.bincount(labels_subset, minlength=num_classes).float().clamp(min=1.0)
    n = labels_subset.size(0)
    weights = n / (num_classes * counts)
    return weights.to(device)


def lr_lambda(epoch, total_epochs, warmup_frac=0.1):
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))


def train_one_final_run(
    src_name, model_type, all_data, device, seed,
    epochs=30, lr=1e-3, weight_decay=1e-4, hidden=256, dropout=0.3,
    grad_clip_norm=1.0, collapse_epoch_start=3, collapse_threshold=0.98,
):
    """
    model_type in {"tbase", "hybrid"}. Always class-weighted (established as
    necessary), always grad-clipped + LR-scheduled. Trains on the NATURAL
    (unmodified) train split -- this is the real cross-domain benchmark, not
    the synthetic imbalance sweep.
    """
    set_seed(seed)

    data_src = all_data[src_name]
    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    train_mask = data_src["train_mask"].to(device)
    val_mask = data_src["val_mask"].to(device)
    edge_index_src = data_src["edge_index"].to(device)
    num_nodes_src = data_src["num_nodes"]

    class_weights = compute_class_weights(y_src, train_mask, 2, device)

    emb_dim = x_src.size(1)
    if model_type == "hybrid":
        model = HybridGCNTransformer(emb_dim=emb_dim, gcn_hidden=hidden, fusion_hidden=hidden,
                                      num_classes=2, dropout=dropout).to(device)
    elif model_type == "tbase":
        model = TextOnlyClassifier(emb_dim=emb_dim, hidden=hidden, num_classes=2, dropout=dropout).to(device)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: lr_lambda(ep, epochs))
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_f1 = -1.0
    best_state = None
    epoch_first_collapse = None

    def forward(m, x):
        if model_type == "hybrid":
            return m(x, edge_index_src, num_nodes_src)
        return m(x)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = forward(model, x_src)
        loss = criterion(logits[train_mask], y_src[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits_full = forward(model, x_src)
            val_logits = val_logits_full[val_mask]
            val_labels = y_src[val_mask]
            val_pred = val_logits.argmax(dim=-1)
            val_f1 = f1_score(val_labels.cpu().numpy(), val_pred.cpu().numpy(), average="macro")
            pred_real_frac = val_pred.float().mean().item()

        is_collapsed_now = (pred_real_frac >= collapse_threshold) or (pred_real_frac <= (1 - collapse_threshold))
        if is_collapsed_now and epoch >= collapse_epoch_start and epoch_first_collapse is None:
            epoch_first_collapse = epoch

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # ---- evaluate the trained model on ALL FOUR targets' test sets ----
    results = {}
    model.eval()
    for tgt_name, data_tgt in all_data.items():
        x_tgt = data_tgt["emb"].to(device)
        y_tgt = data_tgt["labels"].to(device)
        test_mask_tgt = data_tgt["test_mask"].to(device)
        edge_index_tgt = data_tgt["edge_index"].to(device)
        num_nodes_tgt = data_tgt["num_nodes"]

        with torch.no_grad():
            if model_type == "hybrid":
                logits_tgt = model(x_tgt, edge_index_tgt, num_nodes_tgt)
            else:
                logits_tgt = model(x_tgt)
            logits_tgt = logits_tgt[test_mask_tgt]
            labels_tgt = y_tgt[test_mask_tgt]
            pred_tgt = logits_tgt.argmax(dim=-1)
            acc = accuracy_score(labels_tgt.cpu().numpy(), pred_tgt.cpu().numpy())
            macro_f1 = f1_score(labels_tgt.cpu().numpy(), pred_tgt.cpu().numpy(), average="macro")
            pred_real_frac_tgt = pred_tgt.float().mean().item()

        results[tgt_name] = {"acc": acc, "macro_f1": macro_f1, "pred_real_frac": pred_real_frac_tgt}

    return {
        "source": src_name, "model_type": model_type, "seed": seed,
        "best_val_macro_f1": best_val_f1,
        "epoch_first_collapse": epoch_first_collapse if epoch_first_collapse is not None else -1,
        "target_results": results,
    }


def run_final_cross_domain(
    base_dir, datasets=["liar", "coaid", "fakenewsnet", "fakehealth"],
    seeds=[0, 1, 2], model_name="roberta-base", epochs=30,
    max_length=128, batch_size=32, k=10, dropout=0.3,
):
    device = get_device()
    all_data = prepare_all_datasets(
        base_dir=base_dir, model_name=model_name, max_length=max_length,
        batch_size=batch_size, k=k, device=device, dataset_names=datasets,
    )

    out_dir = os.path.join(base_dir, "final_cross_domain_results")
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    total_runs = len(datasets) * 2 * len(seeds)
    run_counter = 0

    for model_type in ["tbase", "hybrid"]:
        for src_name in datasets:
            for seed in seeds:
                run_counter += 1
                print(f"\n[{run_counter}/{total_runs}] model={model_type} source={src_name} seed={seed}")
                result = train_one_final_run(
                    src_name=src_name, model_type=model_type, all_data=all_data,
                    device=device, seed=seed, epochs=epochs, dropout=dropout,
                )
                print(
                    f"  best_val_F1={result['best_val_macro_f1']:.4f} "
                    f"first_collapse_epoch={result['epoch_first_collapse']}"
                )
                for tgt_name, tgt_res in result["target_results"].items():
                    rows.append({
                        "model_type": model_type, "source": src_name, "seed": seed, "target": tgt_name,
                        "test_macro_f1": tgt_res["macro_f1"], "test_acc": tgt_res["acc"],
                        "test_pred_real_frac": tgt_res["pred_real_frac"],
                        "epoch_first_collapse": result["epoch_first_collapse"],
                    })

    raw_df = pd.DataFrame(rows)
    raw_path = os.path.join(out_dir, "final_cross_domain_raw.csv")
    raw_df.to_csv(raw_path, index=False)

    agg = raw_df.groupby(["model_type", "source", "target"])["test_macro_f1"].agg(["mean", "std"]).reset_index()
    agg_path = os.path.join(out_dir, "final_cross_domain_agg.csv")
    agg.to_csv(agg_path, index=False)

    print(f"\n=== Final cross-domain sweep complete ===")
    print(f"Saved raw per-seed results to: {raw_path}")
    print(f"Saved aggregated (mean/std) results to: {agg_path}")

    for model_type in ["tbase", "hybrid"]:
        print(f"\n=== {model_type.upper()} Mean Test Macro-F1 (Train=row, Test=col) ===")
        sub = agg[agg["model_type"] == model_type]
        pivot_mean = sub.pivot(index="source", columns="target", values="mean").reindex(index=datasets, columns=datasets)
        print(pivot_mean.round(4))
        print(f"\n=== {model_type.upper()} Std Dev Test Macro-F1 (across {len(seeds)} seeds) ===")
        pivot_std = sub.pivot(index="source", columns="target", values="std").reindex(index=datasets, columns=datasets)
        print(pivot_std.round(4))

    tbase_mean = agg[agg["model_type"] == "tbase"].pivot(index="source", columns="target", values="mean").reindex(index=datasets, columns=datasets)
    hybrid_mean = agg[agg["model_type"] == "hybrid"].pivot(index="source", columns="target", values="mean").reindex(index=datasets, columns=datasets)
    delta = (hybrid_mean - tbase_mean)
    delta_path = os.path.join(out_dir, "final_cross_domain_delta.csv")
    delta.to_csv(delta_path)
    print(f"\n=== Delta (Hybrid - T-Base), Mean Test Macro-F1 ===")
    print(delta.round(4))
    print(f"Saved delta table to: {delta_path}")

    n_collapsed = (raw_df["epoch_first_collapse"] >= 0).sum()
    print(f"\nAny collapse detected in this run (should be 0 given class weighting): {n_collapsed} / {len(raw_df)} rows flagged")

    return raw_df, agg


run_final_cross_domain(
    base_dir="drive/MyDrive/CSML/data",
    datasets=["liar", "coaid", "fakenewsnet", "fakehealth"],
    seeds=[0, 1, 2],
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    batch_size=32,
    k=10,
    dropout=0.3,
)
